In [1]:
import os
import shutil

source_dir = '../All_Data_json'
dest_dir = 'All_psi'

if not os.path.exists(dest_dir):
    os.makedirs(dest_dir)

for filename in os.listdir(source_dir):
    if filename.startswith('psi'):
        source_path = os.path.join(source_dir, filename)
        dest_path = os.path.join(dest_dir, filename)
        shutil.copy(source_path, dest_path)
        print(f"copy:{filename}")


copy:psi_Gemma2_27b_zero_shot.json
copy:psi_Gemma2_9b_zero_shot.json
copy:psi_GPT4o_mini_zero_shot.json
copy:psi_GPT4o_zero_shot.json
copy:psi_Llama3_70b_zero_shot.json
copy:psi_Llama3_8b_zero_shot.json
copy:psi_Mistral_7b_zero_shot.json


In [3]:
import os
import json
import numpy as np
import csv
from scipy.stats import pearsonr

ground_truth_path = "../PSI_dataset_scored.json"
pred_dir = "All_psi"
output_csv = "./Output/psi_evaluation_results.csv"

with open(ground_truth_path, "r") as f:
    ground_truth = json.load(f)

traits = ["Extraversion", "Agreeableness", "Neuroticism", "Conscientiousness", "Openness"]
trait_short = ["Ext", "Agr", "Neu", "Con", "Ope"]

rows = []

pred_files = sorted(os.listdir(pred_dir))

for filename in pred_files:
    pred_path = os.path.join(pred_dir, filename)
    
    with open(pred_path, "r") as f:
        preds = json.load(f)
        if isinstance(preds, dict): 
            print(f"⚠️  {filename} not a list, skipping")
            continue

    pred_scores = {trait: [] for trait in traits}
    true_scores = {trait: [] for trait in traits}

    for i in range(len(ground_truth)):
        gt = ground_truth[i]
        pr = preds[i]

        for trait in traits:
            pred_scores[trait].append(pr[trait])
            true_scores[trait].append(gt[trait])

    mae_vals = []
    r_vals = []
    for trait in traits:
        preds_arr = np.array(pred_scores[trait])
        trues_arr = np.array(true_scores[trait])
        mae = np.mean(np.abs(preds_arr - trues_arr))
        r, _ = pearsonr(preds_arr, trues_arr)
        mae_vals.append(mae)
        r_vals.append(r)

    rows.append([filename] + mae_vals + r_vals)

with open(output_csv, "w", newline="") as f:
    writer = csv.writer(f)
    header = ["filename"] + [f"MAE_{s}" for s in trait_short] + [f"r_{s}" for s in trait_short]
    writer.writerow(header)
    writer.writerows(rows)

print(f"{output_csv}")

./Output/psi_evaluation_results.csv
